# AI Learning Assistant — Smart Course Knowledge Platform

**Final Project: Modern Data Engineering for AI Systems**

Pipeline: **Knowledge Base → Data Quality → Chunking → Embeddings → ChromaDB → Semantic Retrieval → RAG-style Answer**

In [ ]:
!pip -q install chromadb sentence-transformers

In [ ]:
import re
import chromadb
from sentence_transformers import SentenceTransformer

knowledge_base = '''Modern Data Engineering for AI Systems — Course Knowledge Base

SECTION: Modern Data Architectures
A modern AI data platform collects, processes, stores, governs, retrieves, and serves data for AI applications.
Common modern architecture concepts include Data Warehouses, Data Lakes, and Data Lakehouses.
A Data Lakehouse combines the flexibility of a Data Lake with reliability features such as schema enforcement and ACID transactions.
Modern platforms often decouple compute from storage so each layer can scale independently.

SECTION: Real-Time Data Pipelines
Real-time data engineering treats data as a continuous stream of events instead of waiting for periodic batch processing.
An event-driven pipeline commonly contains producers, a streaming broker, and a stream processor.
Producers generate events from applications, devices, or systems.
Streaming brokers such as Apache Kafka or Redpanda receive and preserve events.
Stream processors such as Apache Flink or Spark Structured Streaming process events as they arrive.
Real-time pipelines are useful for applications such as fraud detection, live personalization, and location tracking.

SECTION: Vector Databases and Embeddings
An embedding is a numerical vector representation of data such as text, images, or audio.
Embeddings place semantically similar content near each other in vector space.
A vector database stores and searches embedding vectors using similarity rather than exact keyword matching.
Similarity search may use cosine similarity, Euclidean distance, or dot product.
Examples of vector databases include ChromaDB, Pinecone, Qdrant, Weaviate, and Milvus.

SECTION: Retrieval-Augmented Generation (RAG)
RAG combines retrieval with a language model so generated answers are grounded in retrieved context.
A common RAG pipeline is: ingest documents, clean and validate data, split documents into chunks, create embeddings, index vectors, retrieve relevant chunks, then generate an answer from the retrieved context.
Chunking is important because documents need to be divided into meaningful pieces before embedding and retrieval.
The retrieval stage returns the top-k most relevant chunks for a user question.
Grounding responses in retrieved enterprise knowledge can improve accuracy and reduce hallucinations.

SECTION: Data Quality
Data quality is important because poor input data leads to poor AI outputs.
Six core dimensions of data quality are accuracy, completeness, consistency, validity, uniqueness, and timeliness.
Validation techniques include schema validation, range validation, type validation, duplicate detection, missing-value checks, consistency validation, and pattern validation.
Quality gates can block invalid data before it enters production systems.

SECTION: Data Governance and Lineage
Data governance defines policies, ownership, access, security, and compliance rules for data.
Data lineage tracks how data moves and changes across systems.
Governance and lineage help make enterprise AI systems trustworthy, auditable, and transparent.

SECTION: Integrated AI Data Platform
A unified AI data platform can combine data sources, real-time pipelines, quality validation, governance, storage, embedding generation, a vector database, RAG retrieval, and AI applications.
A high-level flow is:
Data Sources -> Data Pipeline -> Data Quality -> Governance -> Storage -> Embeddings -> Vector Database -> RAG -> AI Application.
Recommended technologies mentioned in the course include Kafka, Spark, Flink, Great Expectations, Sentence Transformers, ChromaDB, LangChain, LlamaIndex, and large language models.
'''
print('Knowledge base loaded:', len(knowledge_base), 'characters')

## 1. Automated Data Quality Checks

In [ ]:
def data_quality_check(text):
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    report = {
        'not_empty': bool(text.strip()),
        'minimum_length': len(text.strip()) > 300,
        'has_sections': 'SECTION:' in text,
        'duplicate_lines': len(lines) - len(set(lines))
    }
    report['passed'] = report['not_empty'] and report['minimum_length'] and report['has_sections'] and report['duplicate_lines'] == 0
    return report

quality_report = data_quality_check(knowledge_base)
quality_report

## 2. Chunking

In [ ]:
def chunk_text(text, max_chars=650):
    sections = re.split(r'\n(?=SECTION:)', text)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        sentences = re.split(r'(?<=[.!?])\s+', section)
        current = ''
        for sentence in sentences:
            if len(current) + len(sentence) + 1 <= max_chars:
                current += (' ' if current else '') + sentence
            else:
                if current: chunks.append(current.strip())
                current = sentence
        if current: chunks.append(current.strip())
    return chunks

chunks = chunk_text(knowledge_base)
print('Number of chunks:', len(chunks))
chunks[:2]

## 3. Embeddings + ChromaDB

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks).tolist()

client = chromadb.Client()
collection = client.create_collection('course_knowledge')
collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)
print('Indexed chunks:', collection.count())

## 4. Semantic Retrieval + RAG-style Answer

In [ ]:
def retrieve(query, top_k=3):
    q_emb = model.encode([query]).tolist()
    result = collection.query(query_embeddings=q_emb, n_results=top_k)
    return list(zip(result['documents'][0], result['distances'][0]))

def rag_answer(query, retrieved):
    context = ' '.join([doc for doc, _ in retrieved])
    sentences = re.split(r'(?<=[.!?])\s+', context)
    q_terms = set(re.findall(r'\w+', query.lower()))
    ranked = []
    for s in sentences:
        terms = set(re.findall(r'\w+', s.lower()))
        ranked.append((len(q_terms & terms), s.strip()))
    ranked.sort(key=lambda x: x[0], reverse=True)
    selected = [s for _, s in ranked if s][:3]
    return ' '.join(selected)

question = 'What is a vector database and how is it used in RAG?'
retrieved = retrieve(question)
print('QUESTION:', question)
print('\nANSWER:', rag_answer(question, retrieved))
print('\nTOP RETRIEVED CHUNKS:')
for i, (doc, dist) in enumerate(retrieved, 1):
    print(f'\n{i}. distance={dist:.4f}\n{doc}')

## Architecture

```text
Course Knowledge
      ↓
Data Quality Checks
      ↓
Chunking
      ↓
Sentence-Transformer Embeddings
      ↓
ChromaDB Vector Database
      ↓
Semantic Retrieval (Top-k)
      ↓
Grounded RAG-style Answer
```
